# Cache-Oblivious String Sorting

This notebook starts the implementation workflow for the cache-oblivious algorithms from the report. It sets up the environment, imports the reusable Python modules, defines a first batch of datasets, and runs basic correctness checks.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import platform, subprocess, os, re
import numpy as np

sns.set_theme(style="whitegrid")
# load timing results and cache profiling results (try multiple relative paths)
def _find_existing(paths):
    for p in paths:
        if os.path.exists(p):
            return p
    return None
results_paths = ["Analysis/results.csv", "results.csv", "../results.csv"]
cache_paths = ["Analysis/cache_results.csv", "cache_results.csv", "../cache_results.csv"]
results_file = _find_existing(results_paths)
cache_file = _find_existing(cache_paths)
if results_file is None:
    raise FileNotFoundError("results.csv not found in any of: %s" % results_paths)
if cache_file is None:
    raise FileNotFoundError("cache_results.csv not found in any of: %s" % cache_paths)
results_df = pd.read_csv(results_file)
cache_df = pd.read_csv(cache_file)
# compute average string length S from total_length / count in cache_df
cache_df['avg_string_length'] = cache_df['total_length'] / cache_df['count']

# detect M and B (LLC size and cache line size) if not present in cache_df
def parse_size_str(s):
    s = s.strip()
    if s.endswith(('K','k')):
        return int(float(s[:-1]) * 1024)
    if s.endswith(('M','m')):
        return int(float(s[:-1]) * 1024 * 1024)
    try:
        return int(s)
    except:
        return None

M = cache_df['M'].dropna().iloc[0] if 'M' in cache_df.columns and cache_df['M'].notna().any() else None
B = cache_df['B'].dropna().iloc[0] if 'B' in cache_df.columns and cache_df['B'].notna().any() else None
if M is None or B is None:
    if platform.system() == 'Darwin':
        try:
            M = int(subprocess.check_output(['sysctl','-n','hw.l3cachesize']).strip())
        except Exception:
            M = None
        try:
            B = int(subprocess.check_output(['sysctl','-n','hw.cachelinesize']).strip())
        except Exception:
            B = None
    else:
        try:
            idx3 = '/sys/devices/system/cpu/cpu0/cache/index3/size'
            if os.path.exists(idx3):
                with open(idx3) as f:
                    M = parse_size_str(f.read().strip())
        except Exception:
            M = None
        try:
            path = '/sys/devices/system/cpu/cpu0/cache/index3/coherency_line_size'
            if not os.path.exists(path):
                path = '/sys/devices/system/cpu/cpu0/cache/index0/coherency_line_size'
            if os.path.exists(path):
                with open(path) as f:
                    B = int(f.read().strip())
        except Exception:
            B = None

print('Detected M=', M, 'B=', B)

# extract burstsort threshold from header (fallback to 8192)
burst_threshold = 8192
try:
    with open('cpp/src/algorithms/burstsort.hpp') as f:
        txt = f.read()
        m = re.search(r'default\s*=\s*(\d+)', txt) or re.search(r'threshold\s*=\s*(\d+)', txt)
        if m:
            burst_threshold = int(m.group(1))
except Exception:
    pass

# compute theoretical Q approximations (bytes transferred / B) on timing data
# also compute Q_theory on cache_df for miss comparisons
passes = 1
cache_df['N'] = cache_df['count']
cache_df['S'] = cache_df['avg_string_length']
cache_df['total_chars'] = cache_df['N'] * cache_df['S']
cache_df.loc[cache_df['algorithm']=='msd_radix_sort', 'Q_theory'] = (cache_df['total_chars'] * passes) / (B if B else 1)
cache_df.loc[cache_df['algorithm'].str.contains('burst', na=False), 'Q_theory'] = (cache_df['N'] * cache_df['S'].clip(upper=burst_threshold)) / (B if B else 1)
mask_merge_cache = cache_df['algorithm']=='lazy_funnelsort'
cache_df.loc[mask_merge_cache, 'Q_theory'] = (cache_df.loc[mask_merge_cache, 'total_chars'] / (B if B else 1)) * np.log2(cache_df.loc[mask_merge_cache, 'N'])
mask_std_cache = cache_df['algorithm']=='std::sort'
cache_df.loc[mask_std_cache, 'Q_theory'] = (cache_df.loc[mask_std_cache, 'total_chars'] / (B if B else 1)) * np.log2(cache_df.loc[mask_std_cache, 'N'])

results_df['avg_string_length'] = results_df['total_length'] / results_df['count']
results_df['N'] = results_df['count']
results_df['S'] = results_df['avg_string_length']
results_df['total_chars'] = results_df['N'] * results_df['S']
results_df.loc[results_df['algorithm']=='msd_radix_sort', 'Q_theory'] = (results_df['total_chars'] * passes) / (B if B else 1)
results_df.loc[results_df['algorithm'].str.contains('burst', na=False), 'Q_theory'] = (results_df['N'] * results_df['S'].clip(upper=burst_threshold)) / (B if B else 1)
mask_merge = results_df['algorithm']=='lazy_funnelsort'
results_df.loc[mask_merge, 'Q_theory'] = (results_df.loc[mask_merge, 'total_chars'] / (B if B else 1)) * np.log2(results_df.loc[mask_merge, 'N'])
mask_std = results_df['algorithm']=='std::sort'
results_df.loc[mask_std, 'Q_theory'] = (results_df.loc[mask_std, 'total_chars'] / (B if B else 1)) * np.log2(results_df.loc[mask_std, 'N'])

# use timing data for plots to avoid NaNs from mismatched cache counts
df = results_df
df.head()


## 1. Set Up Project Environment

## 3. Define Core Configuration

## 5. Add Basic Validation Checks

## 7. Summarize Results


In [ ]:
plt.figure(figsize=(10, 6))
sns.lineplot(data=df[df["dataset"] == "random"], x="count", y="elapsed_seconds", hue="algorithm", marker="o")
plt.title("Sorting Performance (Synthetic Random)")
plt.ylabel("Time (Seconds)")
plt.xlabel("Number of Strings")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.lineplot(data=df[df["dataset"] == "prefix_heavy"], x="count", y="elapsed_seconds", hue="algorithm", marker="o")
plt.title("Sorting Performance (Synthetic Prefix Heavy)")
plt.ylabel("Time (Seconds)")
plt.xlabel("Number of Strings")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.lineplot(data=df[df["dataset"] == "wiki_random"], x="count", y="elapsed_seconds", hue="algorithm", marker="o")
plt.title("Sorting Performance (Wikipedia Random)")
plt.ylabel("Time (Seconds)")
plt.xlabel("Number of Strings")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.lineplot(data=df[df["dataset"] == "wiki_heavy"], x="count", y="elapsed_seconds", hue="algorithm", marker="o")
plt.title("Sorting Performance (Wikipedia Long Titles)")
plt.ylabel("Time (Seconds)")
plt.xlabel("Number of Strings")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
miss_df = cache_df.dropna(subset=["Q_theory", "lld_misses"])
sns.scatterplot(data=miss_df, x="Q_theory", y="lld_misses", hue="algorithm", style="dataset", alpha=0.8)
plt.xscale("log")
plt.yscale("log")
plt.title("Q_theory vs LLC Misses (Cache Profile)")
plt.xlabel("Q_theory (approx cache lines)")
plt.ylabel("LLC Misses")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
miss_df = cache_df.dropna(subset=["lld_misses", "total_length", "count"])
miss_df = miss_df.assign(misses_per_char=miss_df["lld_misses"] / miss_df["total_length"], misses_per_string=miss_df["lld_misses"] / miss_df["count"])
sns.scatterplot(data=miss_df, x="count", y="misses_per_char", hue="algorithm", style="dataset", alpha=0.8)
plt.xscale("log")
plt.yscale("log")
plt.title("LLC Misses per Character (Cache Profile)")
plt.xlabel("Number of Strings")
plt.ylabel("Misses per Character")
plt.show()